In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_test_current_max_S5_20261020_150000.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_test_current_max_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
TRACE_CSV = RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")
CODES = ["vco_tune_coarse", "vco_current_min", "vco_current_max"]
all_runs = pd.read_csv(RESULT_CSV)
traces = pd.read_csv(TRACE_CSV) if TRACE_CSV.exists() else pd.DataFrame(columns=CODES + ["kind"])
HAS_SCOPE = all_runs["vctrl_meas"].notna().any()

# Operating point of a row: the predicted Vctrl of the earlier tests (current_max_family / _sweep), or "matched |Kvco|"
# for step 3 (current_max_matched: vctrl_target NaN, kvco_request = the measured |Kvco| of the mapped point).
# kvco_plot: the |Kvco| to show, measured (step 3) or predicted by the table (earlier tests).
MATCHED = all_runs["vctrl_target"].isna()
all_runs["op_point"] = np.where(MATCHED, "matched |Kvco|", [f"Vctrl {v:.2f} V" for v in all_runs["vctrl_target"]])
all_runs["kvco_plot"] = np.where(MATCHED, all_runs["kvco_request"], all_runs["kvco_pred"])
FAMILY_COLS = ["op_point", "vco_tune_coarse", "vco_current_min"]  # fixed per family
all_runs = all_runs.sort_values(FAMILY_COLS + ["vco_current_max"]).reset_index(drop=True)
runs = all_runs[(all_runs["locked"].fillna(0) == 1) & (all_runs["status"] == "ok")].reset_index(drop=True)
print(f"{RESULT_CSV.name}: {len(all_runs)} settings, f_vco {all_runs['f_vco'].min() / 1e6:.0f}-"
      f"{all_runs['f_vco'].max() / 1e6:.0f} MHz, operating points {', '.join(all_runs['op_point'].unique())}, "
      f"Vctrl pad {all_runs['vctrl_pad'].iloc[0]}; "
      f"{(all_runs['locked'].fillna(0) == 1).sum()} locked, {len(runs)} measured ok")

pd.DataFrame({
    "max": all_runs["vco_current_max"],
    "f_vco [MHz]": all_runs["f_vco"] / 1e6,
    "setting": [f"c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}"
                for r in all_runs.itertuples()],
    "Vctrl pred [V]": all_runs["vctrl_pred"],
    "Vctrl meas [V]": all_runs["vctrl_meas"],
    "operating point": all_runs["op_point"],
    "|Kvco| [MHz/V]": all_runs["kvco_plot"] / 1e6,
    "locked (pad on)": all_runs["locked_peek"],
    "locked": all_runs["locked"],
    "carrier [dBm]": all_runs["carrier_dbm"],
    "jitter curve [ps]": all_runs["time_jitter_trace_ps"],
    "jitter noise [ps]": all_runs["time_jitter_noise_ps"],
    "jitter spurs [ps]": all_runs["time_jitter_spur_ps"],
    "status": all_runs["status"],
}).round(3)

pll_test_current_max_S5_20261020_200224.csv: 35 settings, f_vco 760-760 MHz, operating points matched |Kvco|, Vctrl pad read; 35 locked, 35 measured ok


,max,f_vco [MHz],setting,Vctrl pred [V],Vctrl meas [V],operating point,|Kvco| [MHz/V],locked (pad on),locked,carrier [dBm],jitter curve [ps],jitter noise [ps],jitter spurs [ps],status
0,0,760.0,c9 min 9 max 0,0.642,0.626,matched |Kvco|,1846.097,1,1,0.352,341.390,201.502,275.579,ok
1,0,760.0,c9 min 9 max 0,0.642,0.626,matched |Kvco|,1846.097,1,1,0.355,331.864,187.569,273.773,ok
2,0,760.0,c9 min 9 max 0,0.642,0.626,matched |Kvco|,1846.097,1,1,0.407,307.582,173.314,254.105,ok
3,0,760.0,c9 min 9 max 0,0.642,0.626,matched |Kvco|,1846.097,1,1,0.409,300.466,173.141,245.564,ok
4,0,760.0,c9 min 9 max 0,0.642,0.627,matched |Kvco|,1846.097,1,1,0.409,309.918,167.650,260.658,ok
5,0,760.0,c9 min 9 max 0,0.642,0.627,matched |Kvco|,1846.097,1,1,0.412,278.564,174.705,216.970,ok
6,0,760.0,c9 min 9 max 0,0.642,0.626,matched |Kvco|,1846.097,1,1,0.397,317.980,191.980,253.486,ok
7,8,760.0,c9 min 9 max 8,0.638,0.619,matched |Kvco|,1702.231,1,1,0.352,356.257,205.158,291.255,ok
8,8,760.0,c9 min 9 max 8,0.638,0.619,matched |Kvco|,1702.231,1,1,0.363,326.390,194.798,261.886,ok
9,8,760.0,c9 min 9 max 8,0.638,0.619,matched |Kvco|,1702.231,1,1,0.408,292.116,178.437,231.283,ok


In [3]:
import sys

sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import bin_phase_noise

INK, GRID = "#52514e", "#e4e3df"
OFFSET_AXIS = dict(type="log", tickvals=[1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8],
                   ticktext=["100 Hz", "1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz", "100 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))
N_BINS = 300  # points per plotted phase-noise curve; None = every measured point
L_OFFSETS = (1e3, 1e4, 1e5, 1e6)  # [Hz] spot levels read from the full curve (power mean within +-10 %)


def max_color(code):
    """Colour of a vco_current_max code (0: most current, blue; 15: least, yellow)."""
    return sample_colorscale("Viridis", [code / 15])[0]


def trace_of(r, kind):
    """Traces of the setting in row r (matched on the VCO codes and f_vco: unique per measured point) of one kind."""
    sel = traces["kind"] == kind
    for c in CODES + ["f_vco"]:
        sel &= traces[c] == getattr(r, c)
    return traces[sel]


def curve_level(curve, f):
    """Level [dBc/Hz] of the curve (columns freq, level_dbm) at offset f: power mean of the points within +-10 %."""
    near = curve[(curve["freq"] >= 0.9 * f) & (curve["freq"] <= 1.1 * f)]
    if near.empty:
        return np.nan
    return 10 * np.log10(np.mean(10 ** (near["level_dbm"] / 10)))


def binned(df, f_start, f_stop):
    """The curve in df (freq, level_dbm) between f_start and f_stop averaged into N_BINS log-spaced bins in linear
    power (pll_util.bin_phase_noise): display only, the jitter uses the measured points."""
    df = df[(df["freq"] >= f_start) & (df["freq"] <= f_stop)].sort_values("freq")
    if N_BINS is None or len(df) < 2:
        return df
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], np.geomspace(f_start, f_stop, N_BINS + 1))
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()


# Phase noise at spot offsets per measured setting, referred to the VCO (pad + 20 log10 N)
for f in L_OFFSETS:
    runs[f"L_vco_{f:.0e}"] = [curve_level(trace_of(r, "L_trace"), f) + 20 * np.log10(r.total_div)
                              for r in runs.itertuples()]

In [4]:
# Performance against vco_current_max, one colour per coarse / min family and operating point (within one only the
# max code and the frequency change). Top left: RMS jitter on the pad (solid: full curve, dashed: noise part). Top
# right: the full curve at 10 kHz (solid) and 1 MHz (dashed), referred to the VCO. Bottom left: Vctrl (filled:
# measured, open: predicted by the table). Bottom right: |Kvco| (measured for step 3, else the table's prediction) and
# f_vco of each setting. Table: rank correlation of the jitter with vco_current_max, Vctrl, |Kvco| and f_vco.
FAMILY_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.1, vertical_spacing=0.14,
                    specs=[[{}, {}], [{}, {"secondary_y": True}]],
                    subplot_titles=("RMS jitter on the pad", "phase noise at the VCO", "Vctrl",
                                    "|Kvco| (markers) and f_vco (line)"))
all_families = all_runs.groupby(FAMILY_COLS, sort=False)
for i, ((op_point, coarse, cur_min), fam_all) in enumerate(all_families):
    color = FAMILY_COLORS[i % len(FAMILY_COLORS)]
    fam = runs[(runs["op_point"] == op_point) & (runs["vco_tune_coarse"] == coarse)
               & (runs["vco_current_min"] == cur_min)]
    label = f"c{coarse:.0f} min {cur_min:.0f}, {op_point}"
    for col, dash, what in (("time_jitter_trace_ps", "solid", "full curve"), ("time_jitter_noise_ps", "dash", "noise")):
        fig.add_scatter(x=fam["vco_current_max"], y=fam[col], mode="lines+markers", name=f"{label}: {what}",
                        legendgroup=label, line=dict(color=color, dash=dash), row=1, col=1,
                        hovertemplate="max %{x}: %{y:.1f} ps<extra>" + f"{label}, {what}" + "</extra>")
    for f, dash in ((1e4, "solid"), (1e6, "dash")):
        fig.add_scatter(x=fam["vco_current_max"], y=fam[f"L_vco_{f:.0e}"], mode="lines+markers", showlegend=False,
                        legendgroup=label, line=dict(color=color, dash=dash), row=1, col=2,
                        hovertemplate="max %{x}: %{y:.1f} dBc/Hz<extra>" + f"{label}, L({f / 1e3:g} kHz)" + "</extra>")
    for col, symbol in (("vctrl_meas", "circle"), ("vctrl_pred", "circle-open")):
        fig.add_scatter(x=fam_all["vco_current_max"], y=fam_all[col], mode="markers", showlegend=False,
                        legendgroup=label, marker=dict(symbol=symbol, size=10, color=color), row=2, col=1,
                        hovertemplate="max %{x}: %{y:.3f} V<extra>" + f"{label}, {col}" + "</extra>")
    fig.add_scatter(x=fam_all["vco_current_max"], y=fam_all["kvco_plot"] / 1e6, mode="markers", showlegend=False,
                    legendgroup=label, row=2, col=2, secondary_y=False,
                    marker=dict(size=10, color=color, symbol=["circle" if l == 1 else "x" for l in fam_all["locked"]]),
                    hovertemplate="max %{x}: %{y:.0f} MHz/V<extra>" + label + "</extra>")
    fig.add_scatter(x=fam_all["vco_current_max"], y=fam_all["f_vco"] / 1e6, mode="lines", showlegend=False,
                    legendgroup=label, line=dict(color=color, width=1), row=2, col=2, secondary_y=True,
                    hovertemplate="max %{x}: %{y:.0f} MHz<extra>" + label + "</extra>")
fig.update_layout(title="Performance against vco_current_max per family and operating point (x: not locked)",
                  template="plotly_white", height=750, legend=dict(orientation="h", y=-0.12))
fig.update_xaxes(title="vco_current_max (0: most current)", dtick=1, gridcolor=GRID)
fig.update_yaxes(gridcolor=GRID)
fig.update_yaxes(title="RMS jitter [ps]", type="log", row=1, col=1)
fig.update_yaxes(title="L(f) at the VCO [dBc/Hz]", row=1, col=2)
fig.update_yaxes(title="Vctrl [V]", row=2, col=1)
fig.update_yaxes(title="|Kvco| [MHz/V]", row=2, col=2, secondary_y=False)
fig.update_yaxes(title="f_vco [MHz]", row=2, col=2, secondary_y=True, showgrid=False)
save(fig, "metrics_vs_current_max")

cols = ["time_jitter_trace_ps", "time_jitter_noise_ps", "time_jitter_spur_ps"] + [f"L_vco_{f:.0e}" for f in L_OFFSETS]
x_cols = ["vco_current_max", "vctrl_pred"] + (["vctrl_meas"] if HAS_SCOPE else []) + ["kvco_plot", "f_vco"]
runs[cols + x_cols].corr(method="spearman").loc[cols, x_cols].round(2)

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_test_current_max_S5_20261020_200224_metrics_vs_current_max.html and .png


,vco_current_max,vctrl_pred,vctrl_meas,kvco_plot,f_vco
time_jitter_trace_ps,0.60,-0.60,-0.59,-0.52,NaN
time_jitter_noise_ps,0.58,-0.58,-0.56,-0.58,NaN
time_jitter_spur_ps,0.58,-0.58,-0.57,-0.47,NaN
L_vco_1e+03,0.40,-0.40,-0.39,-0.50,NaN
L_vco_1e+04,1.00,-1.00,-0.98,-0.70,NaN
L_vco_1e+05,1.00,-1.00,-0.98,-0.70,NaN
L_vco_1e+06,1.00,-1.00,-0.98,-0.70,NaN


In [5]:
F_start, F_stop = 1e3, 20e6  # offset range shown [Hz]

# Full phase-noise curve per setting, referred to the VCO (pad + 20 log10 N), coloured by vco_current_max (blue: 0,
# most current; yellow: 15, least); one line style per coarse / min family and operating point.
fig = go.Figure()
DASHES = ["solid", "dash", "dot", "dashdot"]
family_keys = list(dict.fromkeys(zip(runs["op_point"], runs["vco_tune_coarse"], runs["vco_current_min"])))
for r in runs.itertuples():
    lt = binned(trace_of(r, "L_trace"), F_start, F_stop)
    if lt.empty:
        continue
    vctrl = r.vctrl_meas if not np.isnan(r.vctrl_meas) else r.vctrl_pred
    name = (f"c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}, "
            f"{r.f_vco / 1e6:.0f} MHz, Vctrl {vctrl:.3f} V, |Kvco| {r.kvco_plot / 1e6:.0f} MHz/V: "
            f"{r.time_jitter_trace_ps:.0f} ps")
    family = family_keys.index((r.op_point, r.vco_tune_coarse, r.vco_current_min))
    fig.add_scatter(x=lt["freq"], y=lt["level_dbm"] + 20 * np.log10(r.total_div), mode="lines", name=name,
                    line=dict(color=max_color(r.vco_current_max), width=2, dash=DASHES[family % len(DASHES)]),
                    hovertemplate="%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
fig.update_layout(title="Phase noise referred to the VCO per setting (colour: max code, line style: family)",
                  template="plotly_white", height=600, legend=dict(orientation="h", y=-0.18),
                  hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **OFFSET_AXIS,
                 range=[np.log10(F_start), np.log10(F_stop)])
fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "phase_noise")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_test_current_max_S5_20261020_200224_phase_noise.html and .png


In [ ]:
# Loop-gain sweep (step 3b, set_current varied): jitter against set_current x measured |Kvco| (binary-weighted Icp:
# loop gain ~ code x Kvco), one colour per max code. On one curve: loop gain; a code above the rest: the VCO.
if runs["set_current"].nunique() > 1:
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1, subplot_titles=("total (full curve)", "noise part"))
    for code, grp in runs.groupby("vco_current_max"):
        grp = grp.assign(gain=grp["set_current"] * grp["kvco_plot"] / 1e9).sort_values("gain")
        for c, col in enumerate(("time_jitter_trace_ps", "time_jitter_noise_ps"), 1):
            fig.add_scatter(x=grp["gain"], y=grp[col], mode="lines+markers", name=f"max {code:.0f}",
                            legendgroup=f"max {code:.0f}", showlegend=c == 1, row=1, col=c,
                            line=dict(color=max_color(code)), text=grp["set_current"],
                            hovertemplate="Icp %{text}, gain %{x:.1f}: %{y:.0f} ps<extra>" + f"max {code:.0f}</extra>")
    fig.update_layout(title="RMS jitter against loop gain per vco_current_max", template="plotly_white", height=450)
    fig.update_xaxes(title="set_current x |Kvco| [code GHz/V]", gridcolor=GRID)
    fig.update_yaxes(title="RMS jitter [ps]", gridcolor=GRID)
    save(fig, "jitter_vs_loop_gain")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_test_current_max_S5_20261020_200224_jitter_vs_loop_gain.html and .png


: 